# Build one Student Agent — then keep upgrading it

You will begin with a useful weekly planner, then give the same agent private course knowledge, revision tools and specialist support. The tools are already implemented so you can focus on instructions, decisions and traces.

> All deadlines, calendars, progress records, course notes and actions in this notebook are workshop simulations. The agent may support planning and learning, but it must not produce assessed work for submission.

## Setup

Run these cells once. The root `.env` is used only by this local notebook/runtime. If no key is present, the next cell asks for one without writing it into the notebook.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
from getpass import getpass
from dotenv import load_dotenv
import os

load_dotenv('.env', override=True)
if not os.getenv('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass('Anthropic API key: ')

from workshopkit import *

---
## Mission 1 — Build the Student Planner

**Goal:** `Plan what I should work on this week.`

The agent should inspect deadlines, current progress, fixed commitments and realistic capacity before proposing a plan. It must protect buffer time and ask before saving anything.

In [ ]:
# Inspect the five tool contracts before writing the agent instructions.
for tool in PLANNER_TOOLS:
    print(f'\n{tool.name}: {tool.description}')
    print(tool.input_schema)

In [ ]:
# YOUR TURN — edit the CRAFT instructions and goal, then run the cell.
AGENT_INSTRUCTIONS = """
C — Context: You support a university student using fictional workshop deadline, calendar and progress data.
R — Request: Work out what the student should focus on this week and create a realistic plan.
A — Approach: Fetch missing planning facts with tools, prioritise urgency and remaining workload, and ask rather than guess when critical information is absent.
F — Format & constraints: Return priorities, a day-by-day plan, risks and what to start tonight. Protect rest, include buffer, do not write assessed content, and do not save anything without approval.
T — Test: Every deadline is addressed, scheduled work fits available hours, progress changes the priorities, buffer remains, and all simulated data is labelled.
"""

STUDENT_GOAL = "Plan what I should work on this week."

planner_result = run_student_planner(AGENT_INSTRUCTIONS, STUDENT_GOAL)
show_trace(planner_result, 'MISSION 1 — STUDENT PLANNER TRACE')

### Trace checkpoint

Check the observable evidence:

- Which facts did the agent fetch instead of inventing?
- Did current progress change the priority order?
- Did the work fit the available blocks and leave buffer?
- Did it avoid `save_study_plan()` unless the student approved an action?
- What would you change in CRAFT before changing any code?

---
## Mission 2 — Upgrade the same agent with private course knowledge

The planner can organise time, but it does not automatically know a unit. Add retrieval, mastery and quiz tools so it can answer: **I have 45 minutes. What should I revise?**

In [ ]:
for tool in STUDY_UPGRADE_TOOLS:
    print(f'\n{tool.name}: {tool.description}')

print('\nDirect retrieval check:')
print(search_course_notes.execute({
    'query': 'When should I use Dijkstra instead of breadth-first search?',
    'unit_code': 'CITS2200',
    'top_k': 2,
}))

In [ ]:
# YOUR TURN — retain the planner boundaries and add revision behaviour.
UPGRADED_INSTRUCTIONS = AGENT_INSTRUCTIONS + """

For revision decisions, inspect the mastery record, retrieve relevant supplied notes, and choose a focused explanation or source-grounded quiz. Name the files used. If the documents do not answer the question, say so. Never fabricate course content or assessed solutions.
"""

REVISION_GOAL = "I have 45 minutes. What should I revise for CITS2200, and can you quiz me on the weakest relevant topic?"

revision_result = run_revision_upgrade(UPGRADED_INSTRUCTIONS, REVISION_GOAL)
show_trace(revision_result, 'MISSION 2 — KNOWLEDGE UPGRADE TRACE')

### Retrieval checkpoint

The agent should search only when unit-specific knowledge is needed, cite the returned filename, ground quiz questions in the retrieved passage and avoid pretending the simulated documents are official or complete.

---
## Mission 3 — Let the Student Agent delegate

Specialists are an upgrade, not a new story. The Student Agent may delegate distinct work to a **Researcher**, **Planner** or **Reviewer**, then combine only the useful outputs. One careful call remains a valid choice when delegation adds no value.

In [ ]:
for specialist in SPECIALISTS:
    print(f'{specialist.name}: {specialist.description}')

In [ ]:
# YOUR TURN — decide which specialist calls are genuinely justified.
MANAGER_INSTRUCTIONS = """
You are the same Student Agent, now able to delegate clearly separated work. Use the Researcher to compare supplied evidence, the Planner for sequencing, and the Reviewer to test a student's own draft against criteria. Do not call every specialist by default. Do not produce assessed prose. Clearly label specialist contributions and unresolved gaps.
"""

MANAGER_TASK = """
I have supplied my own outline and the sample CITS2200 rubric. Decide whether specialist help is justified, then give me a revision plan and questions I should answer myself.

My outline: I will compare breadth-first search and Dijkstra, explain their assumptions, then test both on one weighted graph.
Rubric: select an appropriate algorithm, explain why, state assumptions, analyse complexity, and test an edge case.
"""

manager_result = run_manager(MANAGER_INSTRUCTIONS, MANAGER_TASK)
show_trace(manager_result, 'MISSION 3 — SPECIALIST DELEGATION TRACE')

### Architecture checkpoint

Compare the value of delegation with its extra calls, latency and failure points. A multi-agent design is useful only when the specialist boundaries make the result more reliable or easier to inspect.

---
## Make the Student Agent yours

Choose a useful direction: Study Planner, Research Assistant, Revision Coach, Assignment Reviewer, Group Project Coordinator, Career / Application Assistant, or something else. Define the goal, necessary context, tools, retrieval, specialist roles and approval boundaries before adding code.

In [ ]:
MY_AGENT = {
    'goal': '...',
    'context_needed': ['...'],
    'tools_needed': ['...'],
    'retrieval_needed': False,
    'specialist_needed': False,
    'human_approval_before': ['...'],
    'success_test': ['...'],
}

MY_AGENT